Import google drive and mount it at /content/drive

In [1]:
from google.colab import drive # type: ignore[import-not-found]
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile

image_zip = "/content/drive/MyDrive/Bark Images and Labels/BARK_UGA_PICS.zip"
extract_dir = "/content/bark_images"

with zipfile.ZipFile(image_zip, "r") as z:
    image_count = sum(
        1 for item in z.infolist()
        if not item.is_dir() and item.filename.lower().endswith(".jpg")
    )
    names = z.namelist()
    z.extractall(extract_dir)

print(image_count)
print(names[:10])

2764
['BARK_UGA_PICS/', 'BARK_UGA_PICS/ATT1000_Photo 1.jpg', 'BARK_UGA_PICS/ATT1001_Photo 3.jpg', 'BARK_UGA_PICS/ATT1002_Photo 1.jpg', 'BARK_UGA_PICS/ATT1003_Photo 2.jpg', 'BARK_UGA_PICS/ATT1004_Photo 4.jpg', 'BARK_UGA_PICS/ATT1005_Photo 3.jpg', 'BARK_UGA_PICS/ATT1006_Photo 4.jpg', 'BARK_UGA_PICS/ATT1007_Photo 1.jpg', 'BARK_UGA_PICS/ATT1008_Photo 2.jpg']


In [3]:
import pandas as pd

csv_path = "/content/drive/MyDrive/Bark Images and Labels/bark_class_labels.csv"
df = pd.read_csv(csv_path)

photo_cols = ["Photo0", "Photo1", "Photo2", "Photo3"]

long_df = df.melt(
    id_vars=[col for col in df.columns if col not in photo_cols],
    value_vars=photo_cols,
    var_name="photo_slot",
    value_name="image_filename"
)

long_df = long_df.dropna(subset=["image_filename"])

print(df.columns.tolist())
print(long_df.columns.tolist())

['OBJECTID', 'Species', 'Moisture_Condition', 'DBH_in', 'Note', 'CreatedBy', 'CreateTime', 'EditedBy', 'EditTime', 'Photo0', 'Photo1', 'Photo2', 'Photo3']
['OBJECTID', 'Species', 'Moisture_Condition', 'DBH_in', 'Note', 'CreatedBy', 'CreateTime', 'EditedBy', 'EditTime', 'photo_slot', 'image_filename']


In [4]:
import os
import zipfile

# inspect image zip and copy the name of each image file to zip_files. Excludes folders
with zipfile.ZipFile(image_zip, "r") as z:
    zip_files = [name for name in z.namelist() if not name.endswith("/")]

# dictionary: key = image file base name | value = full file path
zip_lookup = {os.path.basename(name): name for name in zip_files}

# create set of all filenames from csv and image zip base names
csv_filenames = set(long_df["image_filename"].astype(str))
zip_basenames = set(zip_lookup.keys())

# subtract image zip names from csv file names and csv file names from image zip names
missing_from_zip = csv_filenames - zip_basenames
extra_in_zip = zip_basenames - csv_filenames

print("Images listed in CSV:", len(csv_filenames))
print("Images in zip:", len(zip_basenames))
print("Missing from zip:", len(missing_from_zip))
print("Extra in zip:", len(extra_in_zip))

print("Images missing:", list(missing_from_zip)[:10])
print("Images extra:", list(extra_in_zip)[:10])

Images listed in CSV: 2764
Images in zip: 2764
Missing from zip: 0
Extra in zip: 0
Images missing: []
Images extra: []


In [5]:
long_df["image_path"] = long_df["image_filename"].map(zip_lookup)
display(long_df.head(20))

unmatched = long_df[long_df["image_path"].isna()]
print("Unmatched rows:", len(unmatched))
display(unmatched.head())

,OBJECTID,Species,Moisture_Condition,DBH_in,Note,CreatedBy,CreateTime,EditedBy,EditTime,photo_slot,image_filename,image_path
0,1,Pin oak,Dry,17.0,NaN,tdb96463_USG,2024/02/05 18:42:18.856+00,tdb96463_USG,2024/02/05 18:42:18.856+00,Photo0,ATT1_Photo 3.jpg,BARK_UGA_PICS/ATT1_Photo 3.jpg
1,2,Eastern white pine,Dry,29.0,NaN,tdb96463_USG,2024/02/05 18:49:21.604+00,tdb96463_USG,2024/02/05 18:49:21.604+00,Photo0,ATT5_Photo 3.jpg,BARK_UGA_PICS/ATT5_Photo 3.jpg
2,3,Yellow-poplar,Dry,18.0,NaN,tdb96463_USG,2024/02/05 18:50:52.876+00,tdb96463_USG,2024/02/05 18:50:52.876+00,Photo0,ATT9_Photo 2.jpg,BARK_UGA_PICS/ATT9_Photo 2.jpg
3,4,Loblolly pine,Dry,16.0,NaN,tdb96463_USG,2024/03/08 20:53:15.938+00,tdb96463_USG,2024/03/08 20:53:15.938+00,Photo0,ATT13_Photo 2.jpg,BARK_UGA_PICS/ATT13_Photo 2.jpg
4,5,Loblolly pine,Dry,17.0,NaN,tdb96463_USG,2024/03/08 20:56:20.335+00,tdb96463_USG,2024/03/08 20:56:20.335+00,Photo0,ATT17_Photo 1.jpg,BARK_UGA_PICS/ATT17_Photo 1.jpg
5,6,Loblolly pine,Wet,16.0,NaN,tdb96463_USG,2024/03/09 19:25:28.257+00,tdb96463_USG,2024/03/09 19:25:28.257+00,Photo0,ATT21_Photo 1.jpg,BARK_UGA_PICS/ATT21_Photo 1.jpg
6,7,Loblolly pine,Wet,12.0,NaN,tdb96463_USG,2024/03/09 19:27:20.191+00,tdb96463_USG,2024/03/09 19:27:20.191+00,Photo0,ATT25_Photo 4.jpg,BARK_UGA_PICS/ATT25_Photo 4.jpg
7,8,Loblolly pine,Wet,11.0,NaN,tdb96463_USG,2024/03/09 19:30:52.069+00,tdb96463_USG,2024/03/09 19:30:52.069+00,Photo0,ATT29_Photo 3.jpg,BARK_UGA_PICS/ATT29_Photo 3.jpg
8,9,Loblolly pine,Wet,18.0,NaN,tdb96463_USG,2024/03/09 19:33:50.921+00,tdb96463_USG,2024/03/09 19:33:50.921+00,Photo0,ATT33_Photo 4.jpg,BARK_UGA_PICS/ATT33_Photo 4.jpg
9,10,Loblolly pine,Wet,19.0,NaN,tdb96463_USG,2024/03/09 19:36:47.021+00,tdb96463_USG,2024/03/09 19:36:47.021+00,Photo0,ATT37_Photo 1.jpg,BARK_UGA_PICS/ATT37_Photo 1.jpg


Unmatched rows: 0


,OBJECTID,Species,Moisture_Condition,DBH_in,Note,CreatedBy,CreateTime,EditedBy,EditTime,photo_slot,image_filename,image_path


In [6]:
pd.set_option("display.max_colwidth", None)   # do not truncate cell contents
pd.set_option("display.max_columns", None)    # show all columns
pd.set_option("display.width", None)          # avoid wrapping based on terminal width


long_df["image_path"] = long_df["image_filename"].apply(
    lambda p: f"{extract_dir}/BARK_UGA_PICS/{p}"
)

display(long_df[["image_filename", "image_path"]].head())
display(long_df.head())

,image_filename,image_path
0,ATT1_Photo 3.jpg,/content/bark_images/BARK_UGA_PICS/ATT1_Photo 3.jpg
1,ATT5_Photo 3.jpg,/content/bark_images/BARK_UGA_PICS/ATT5_Photo 3.jpg
2,ATT9_Photo 2.jpg,/content/bark_images/BARK_UGA_PICS/ATT9_Photo 2.jpg
3,ATT13_Photo 2.jpg,/content/bark_images/BARK_UGA_PICS/ATT13_Photo 2.jpg
4,ATT17_Photo 1.jpg,/content/bark_images/BARK_UGA_PICS/ATT17_Photo 1.jpg


,OBJECTID,Species,Moisture_Condition,DBH_in,Note,CreatedBy,CreateTime,EditedBy,EditTime,photo_slot,image_filename,image_path
0,1,Pin oak,Dry,17.0,NaN,tdb96463_USG,2024/02/05 18:42:18.856+00,tdb96463_USG,2024/02/05 18:42:18.856+00,Photo0,ATT1_Photo 3.jpg,/content/bark_images/BARK_UGA_PICS/ATT1_Photo 3.jpg
1,2,Eastern white pine,Dry,29.0,NaN,tdb96463_USG,2024/02/05 18:49:21.604+00,tdb96463_USG,2024/02/05 18:49:21.604+00,Photo0,ATT5_Photo 3.jpg,/content/bark_images/BARK_UGA_PICS/ATT5_Photo 3.jpg
2,3,Yellow-poplar,Dry,18.0,NaN,tdb96463_USG,2024/02/05 18:50:52.876+00,tdb96463_USG,2024/02/05 18:50:52.876+00,Photo0,ATT9_Photo 2.jpg,/content/bark_images/BARK_UGA_PICS/ATT9_Photo 2.jpg
3,4,Loblolly pine,Dry,16.0,NaN,tdb96463_USG,2024/03/08 20:53:15.938+00,tdb96463_USG,2024/03/08 20:53:15.938+00,Photo0,ATT13_Photo 2.jpg,/content/bark_images/BARK_UGA_PICS/ATT13_Photo 2.jpg
4,5,Loblolly pine,Dry,17.0,NaN,tdb96463_USG,2024/03/08 20:56:20.335+00,tdb96463_USG,2024/03/08 20:56:20.335+00,Photo0,ATT17_Photo 1.jpg,/content/bark_images/BARK_UGA_PICS/ATT17_Photo 1.jpg


In [7]:
path_exists_df = long_df["image_path"].apply(os.path.exists)

display(path_exists_df.head())
print(path_exists_df.value_counts())

,image_path
0,True
1,True
2,True
3,True
4,True


image_path
True    2764
Name: count, dtype: int64


In [8]:
import torch
import torchvision
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("PyTorch version: ", torch.__version__)
print("PyTorch: ", torchvision.__version__)
print("CUDA is available: ", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

label_encoder = LabelEncoder()
long_df["label"] = label_encoder.fit_transform(long_df["Species"])

num_classes = len(label_encoder.classes_)

unique_trees = long_df[["OBJECTID", "Species"]].drop_duplicates()
assert unique_trees["OBJECTID"].is_unique, "Some OBJECTIDs appear with multiple species."

print("Num tree image rows", len(long_df))
print("Num unique trees", len(unique_trees))
print("Number of species/classes", num_classes)

display(unique_trees.head())
display(unique_trees["Species"].value_counts())

train_trees, temp_trees = train_test_split(
    unique_trees,
    test_size=0.30,
    random_state=0,
    stratify=unique_trees["Species"]
)

val_trees, test_trees = train_test_split(
    temp_trees,
    test_size=0.50,
    random_state=0,
)

train_df = long_df[long_df["OBJECTID"].isin(train_trees['OBJECTID'])].copy()
val_df = long_df[long_df["OBJECTID"].isin(val_trees['OBJECTID'])].copy()
test_df = long_df[long_df["OBJECTID"].isin(test_trees['OBJECTID'])].copy()

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("Train trees:", train_df["OBJECTID"].nunique())
print("Validation trees:", val_df["OBJECTID"].nunique())
print("Test trees:", test_df["OBJECTID"].nunique())

train_ids = set(train_df["OBJECTID"])
val_ids = set(val_df["OBJECTID"])
test_ids = set(test_df["OBJECTID"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("No tree leakage detected.")

PyTorch version:  2.10.0+cu128
PyTorch:  0.25.0+cu128
CUDA is available:  True
Using device: cuda
Num tree image rows 2764
Num unique trees 691
Number of species/classes 26


,OBJECTID,Species
0,1,Pin oak
1,2,Eastern white pine
2,3,Yellow-poplar
3,4,Loblolly pine
4,5,Loblolly pine


,count
Species,
Loblolly pine,172
Southern red oak,54
Willow oak,53
Live oak,50
Water oak,35
Scarlet oak,31
Swamp chestnut oak,29
Mockernut hickory,27
Pecan,22


Train images: 1932
Validation images: 416
Test images: 416
Train trees: 483
Validation trees: 104
Test trees: 104
No tree leakage detected.


In [9]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import (
    EfficientNet_B3_Weights,
    ResNet50_Weights,
    MobileNet_V3_Small_Weights
)

def build_model(model_name, pretrained, num_classes):
    if model_name == "efficientnet_b3":
        weights = EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b3(weights=weights)

        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "resnet50":
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)

        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "mobilenet_v3_small":
        weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.mobilenet_v3_small(weights=weights)

        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return model

In [10]:
for model_name in ["efficientnet_b3", "resnet50", "mobilenet_v3_small"]:
    model = build_model(
        model_name=model_name,
        pretrained=True,
        num_classes=num_classes
    )

    print("Model:", model_name)

    if model_name == "resnet50":
        print(model.fc)
    else:
        print(model.classifier)

    print(model_name, "built successfully\n")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 142MB/s] 


Model: efficientnet_b3
Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1536, out_features=26, bias=True)
)
efficientnet_b3 built successfully

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 153MB/s] 


Model: resnet50
Linear(in_features=2048, out_features=26, bias=True)
resnet50 built successfully

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 75.8MB/s]

Model: mobilenet_v3_small
Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=26, bias=True)
)
mobilenet_v3_small built successfully



In [11]:
experiments = [
    ("efficientnet_b3", True),
    ("efficientnet_b3", False),

    ("resnet50", True),
    ("resnet50", False),

    ("mobilenet_v3_small", True),
    ("mobilenet_v3_small", False),
]

In [12]:
from PIL import Image
from torch.utils.data import Dataset
import torch

class BarkDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = int(row["label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [13]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [14]:
from torch.utils.data import DataLoader

batch_size = 32

train_dataset = BarkDataset(train_df, transform=train_transform)
val_dataset = BarkDataset(val_df, transform=eval_transform)
test_dataset = BarkDataset(test_df, transform=eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 61
Validation batches: 13
Test batches: 13


In [15]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Min label:", labels.min().item())
print("Max label:", labels.max().item())

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64
Min label: 0
Max label: 24


In [16]:
model = build_model(
    model_name="mobilenet_v3_small",
    pretrained=True,
    num_classes=num_classes
)

model = model.to(device)

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Output shape:", outputs.shape)
print("Expected shape:", (images.shape[0], num_classes))

Output shape: torch.Size([32, 26])
Expected shape: (32, 26)


In [17]:
from sklearn.metrics import accuracy_score, f1_score
import torch.optim as optim

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Track total loss
        running_loss += loss.item() * images.size(0)

        # Convert model outputs into predicted class numbers
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)

    epoch_accuracy = accuracy_score(all_labels, all_preds)

    epoch_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return epoch_loss, epoch_accuracy, epoch_macro_f1


def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass only; no training update
            outputs = model(images)

            # Compute loss
            loss = criterion(outputs, labels)

            # Track total loss
            running_loss += loss.item() * images.size(0)

            # Convert model outputs into predicted class numbers
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)

    epoch_accuracy = accuracy_score(all_labels, all_preds)

    epoch_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return epoch_loss, epoch_accuracy, epoch_macro_f1

In [18]:
# One-epoch smoke test with the smallest Purdue model
# Goal: verify that training + validation works before running all 6 experiments

model = build_model(
    model_name="mobilenet_v3_small",
    pretrained=True,
    num_classes=num_classes
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print("Starting smoke test...")
print("Model: mobilenet_v3_small")
print("Pretrained: True")
print("Device:", device)

train_loss, train_acc, train_macro_f1 = train_one_epoch(
    model=model,
    dataloader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device
)

val_loss, val_acc, val_macro_f1 = evaluate(
    model=model,
    dataloader=val_loader,
    criterion=criterion,
    device=device
)

print("\nSmoke test complete.")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Train Macro F1: {train_macro_f1:.4f}")

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Validation Macro F1: {val_macro_f1:.4f}")

Starting smoke test...
Model: mobilenet_v3_small
Pretrained: True
Device: cuda

Smoke test complete.
Train Loss: 2.5946
Train Accuracy: 0.3038
Train Macro F1: 0.0680
Validation Loss: 2.3316
Validation Accuracy: 0.4014
Validation Macro F1: 0.1029


In [ ]:
import os
import gc
import time
import pandas as pd

# Folder where checkpoints and results will be saved
output_dir = "/content/drive/MyDrive/bark_model_runs"
os.makedirs(output_dir, exist_ok=True)

num_epochs = 10
learning_rate = 1e-4

all_results = []

print("Starting full experiment run...")
print("Device:", device)
print("Number of epochs per model:", num_epochs)
print("Output directory:", output_dir)

if device.type == "cpu":
    print("\nWARNING: You are training on CPU. This will be very slow.")
    print("For Colab, use: Runtime → Change runtime type → GPU\n")

for model_name, pretrained in experiments:
    weights_label = "imagenet" if pretrained else "random"

    print("=" * 80)
    print(f"Training model: {model_name} | weights: {weights_label}")
    print("=" * 80)

    model = build_model(
        model_name=model_name,
        pretrained=pretrained,
        num_classes=num_classes
    )

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_macro_f1 = -1.0
    best_epoch = 0

    checkpoint_path = os.path.join(
        output_dir,
        f"{model_name}_{weights_label}_best.pth"
    )

    start_time = time.time()

    for epoch in range(num_epochs):
        train_loss, train_acc, train_macro_f1 = train_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device
        )

        val_loss, val_acc, val_macro_f1 = evaluate(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=device
        )

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train Macro F1: {train_macro_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val Macro F1: {val_macro_f1:.4f}"
        )

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch + 1

            torch.save(
                {
                    "model_name": model_name,
                    "pretrained": pretrained,
                    "weights_label": weights_label,
                    "num_classes": num_classes,
                    "class_names": label_encoder.classes_.tolist(),
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": best_epoch,
                    "best_val_macro_f1": best_val_macro_f1,
                },
                checkpoint_path
            )

            print(f"Saved new best checkpoint: {checkpoint_path}")

    training_time_minutes = (time.time() - start_time) / 60

    # Load the best validation checkpoint before final test evaluation
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    test_loss, test_acc, test_macro_f1 = evaluate(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device
    )

    result = {
        "model_name": model_name,
        "weights": weights_label,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_macro_f1": test_macro_f1,
        "training_time_minutes": training_time_minutes,
        "checkpoint_path": checkpoint_path
    }

    all_results.append(result)

    results_df = pd.DataFrame(all_results)
    display(results_df)

    results_csv_path = os.path.join(output_dir, "experiment_results.csv")
    results_df.to_csv(results_csv_path, index=False)

    # Clear memory before next model
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Finished all experiments.")

final_results_df = pd.DataFrame(all_results)
display(final_results_df)

final_results_path = os.path.join(output_dir, "experiment_results.csv")
final_results_df.to_csv(final_results_path, index=False)

print("Saved results to:", final_results_path)

Starting full experiment run...
Device: cuda
Number of epochs per model: 10
Output directory: /content/drive/MyDrive/bark_model_runs
Training model: efficientnet_b3 | weights: imagenet
Epoch 1/10 | Train Loss: 2.8121 | Train Acc: 0.3059 | Train Macro F1: 0.0840 | Val Loss: 2.3026 | Val Acc: 0.4303 | Val Macro F1: 0.1078
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/efficientnet_b3_imagenet_best.pth
Epoch 2/10 | Train Loss: 2.0025 | Train Acc: 0.4596 | Train Macro F1: 0.1588 | Val Loss: 1.7635 | Val Acc: 0.5505 | Val Macro F1: 0.2545
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/efficientnet_b3_imagenet_best.pth
Epoch 3/10 | Train Loss: 1.4721 | Train Acc: 0.6056 | Train Macro F1: 0.3370 | Val Loss: 1.3656 | Val Acc: 0.6106 | Val Macro F1: 0.3178
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/efficientnet_b3_imagenet_best.pth
Epoch 4/10 | Train Loss: 1.0370 | Train Acc: 0.7205 | Train Macro F1: 0.4929 | Val Loss: 1.1360 | Val Ac

,model_name,weights,best_epoch,best_val_macro_f1,test_loss,test_accuracy,test_macro_f1,training_time_minutes,checkpoint_path
0,efficientnet_b3,imagenet,9,0.561245,0.646031,0.829327,0.68841,10.839413,/content/drive/MyDrive/bark_model_runs/efficientnet_b3_imagenet_best.pth


Training model: efficientnet_b3 | weights: random
Epoch 1/10 | Train Loss: 3.1018 | Train Acc: 0.1392 | Train Macro F1: 0.0314 | Val Loss: 3.0702 | Val Acc: 0.2500 | Val Macro F1: 0.0160
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/efficientnet_b3_random_best.pth
Epoch 2/10 | Train Loss: 2.8693 | Train Acc: 0.2371 | Train Macro F1: 0.0195 | Val Loss: 2.9045 | Val Acc: 0.2308 | Val Macro F1: 0.0151
Epoch 3/10 | Train Loss: 2.8078 | Train Acc: 0.2407 | Train Macro F1: 0.0206 | Val Loss: 2.8453 | Val Acc: 0.2716 | Val Macro F1: 0.0326
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/efficientnet_b3_random_best.pth
Epoch 4/10 | Train Loss: 2.7589 | Train Acc: 0.2474 | Train Macro F1: 0.0239 | Val Loss: 2.7739 | Val Acc: 0.2548 | Val Macro F1: 0.0210
Epoch 5/10 | Train Loss: 2.7501 | Train Acc: 0.2505 | Train Macro F1: 0.0238 | Val Loss: 2.8189 | Val Acc: 0.2524 | Val Macro F1: 0.0211
Epoch 6/10 | Train Loss: 2.7104 | Train Acc: 0.2516 | Train Macro F1:

,model_name,weights,best_epoch,best_val_macro_f1,test_loss,test_accuracy,test_macro_f1,training_time_minutes,checkpoint_path
0,efficientnet_b3,imagenet,9,0.561245,0.646031,0.829327,0.688410,10.839413,/content/drive/MyDrive/bark_model_runs/efficientnet_b3_imagenet_best.pth
1,efficientnet_b3,random,10,0.134772,2.136912,0.394231,0.188665,10.585948,/content/drive/MyDrive/bark_model_runs/efficientnet_b3_random_best.pth


Training model: resnet50 | weights: imagenet
Epoch 1/10 | Train Loss: 2.5209 | Train Acc: 0.3339 | Train Macro F1: 0.0788 | Val Loss: 2.0319 | Val Acc: 0.4784 | Val Macro F1: 0.1495
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/resnet50_imagenet_best.pth
Epoch 2/10 | Train Loss: 1.4995 | Train Acc: 0.5782 | Train Macro F1: 0.3106 | Val Loss: 1.2881 | Val Acc: 0.6875 | Val Macro F1: 0.4257
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/resnet50_imagenet_best.pth
Epoch 3/10 | Train Loss: 0.8224 | Train Acc: 0.7723 | Train Macro F1: 0.5611 | Val Loss: 1.0686 | Val Acc: 0.7260 | Val Macro F1: 0.5049
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/resnet50_imagenet_best.pth
Epoch 4/10 | Train Loss: 0.5174 | Train Acc: 0.8494 | Train Macro F1: 0.6831 | Val Loss: 0.9770 | Val Acc: 0.7524 | Val Macro F1: 0.5235
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/resnet50_imagenet_best.pth
Epoch 5/10 | Train Loss: 0.3278 | T

...

Training model: mobilenet_v3_small | weights: random
Epoch 1/10 | Train Loss: 2.9772 | Train Acc: 0.2448 | Train Macro F1: 0.0174 | Val Loss: 3.1847 | Val Acc: 0.2500 | Val Macro F1: 0.0160
Saved new best checkpoint: /content/drive/MyDrive/bark_model_runs/mobilenet_v3_small_random_best.pth
Epoch 2/10 | Train Loss: 2.7575 | Train Acc: 0.2479 | Train Macro F1: 0.0153 | Val Loss: 3.1070 | Val Acc: 0.2500 | Val Macro F1: 0.0160
Epoch 3/10 | Train Loss: 2.6568 | Train Acc: 0.2557 | Train Macro F1: 0.0246 | Val Loss: 3.0715 | Val Acc: 0.2500 | Val Macro F1: 0.0160
Epoch 4/10 | Train Loss: 2.5471 | Train Acc: 0.2774 | Train Macro F1: 0.0370 | Val Loss: 3.0632 | Val Acc: 0.0577 | Val Macro F1: 0.0044
Epoch 5/10 | Train Loss: 2.3603 | Train Acc: 0.3240 | Train Macro F1: 0.0604 | Val Loss: 3.1277 | Val Acc: 0.0577 | Val Macro F1: 0.0044
Epoch 6/10 | Train Loss: 2.1870 | Train Acc: 0.3675 | Train Macro F1: 0.0998 | Val Loss: 3.2259 | Val Acc: 0.0577 | Val Macro F1: 0.0044
Epoch 7/10 | Train Loss:

,model_name,weights,best_epoch,best_val_macro_f1,test_loss,test_accuracy,test_macro_f1,training_time_minutes,checkpoint_path
0,efficientnet_b3,imagenet,9,0.561245,0.646031,0.829327,0.688410,10.839413,/content/drive/MyDrive/bark_model_runs/efficientnet_b3_imagenet_best.pth
1,efficientnet_b3,random,10,0.134772,2.136912,0.394231,0.188665,10.585948,/content/drive/MyDrive/bark_model_runs/efficientnet_b3_random_best.pth
2,resnet50,imagenet,5,0.596721,0.662297,0.829327,0.684434,11.072792,/content/drive/MyDrive/bark_model_runs/resnet50_imagenet_best.pth
3,resnet50,random,10,0.301282,1.770686,0.500000,0.348432,11.376288,/content/drive/MyDrive/bark_model_runs/resnet50_random_best.pth
4,mobilenet_v3_small,imagenet,10,0.536009,0.918733,0.754808,0.595923,9.211817,/content/drive/MyDrive/bark_model_runs/mobilenet_v3_small_imagenet_best.pth
5,mobilenet_v3_small,random,10,0.050845,3.426087,0.110577,0.038413,9.035605,/content/drive/MyDrive/bark_model_runs/mobilenet_v3_small_random_best.pth


Finished all experiments.


,model_name,weights,best_epoch,best_val_macro_f1,test_loss,test_accuracy,test_macro_f1,training_time_minutes,checkpoint_path
0,efficientnet_b3,imagenet,9,0.561245,0.646031,0.829327,0.688410,10.839413,/content/drive/MyDrive/bark_model_runs/efficientnet_b3_imagenet_best.pth
1,efficientnet_b3,random,10,0.134772,2.136912,0.394231,0.188665,10.585948,/content/drive/MyDrive/bark_model_runs/efficientnet_b3_random_best.pth
2,resnet50,imagenet,5,0.596721,0.662297,0.829327,0.684434,11.072792,/content/drive/MyDrive/bark_model_runs/resnet50_imagenet_best.pth
3,resnet50,random,10,0.301282,1.770686,0.500000,0.348432,11.376288,/content/drive/MyDrive/bark_model_runs/resnet50_random_best.pth
4,mobilenet_v3_small,imagenet,10,0.536009,0.918733,0.754808,0.595923,9.211817,/content/drive/MyDrive/bark_model_runs/mobilenet_v3_small_imagenet_best.pth
5,mobilenet_v3_small,random,10,0.050845,3.426087,0.110577,0.038413,9.035605,/content/drive/MyDrive/bark_model_runs/mobilenet_v3_small_random_best.pth


Saved results to: /content/drive/MyDrive/bark_model_runs/experiment_results.csv


In [20]:
import os

save_dir = "/content/drive/MyDrive/trained_bark_models"
os.makedirs(save_dir, exist_ok=True)